# Araseの電磁場 despun データのチェック → WPT/SGI座標系の時点で擾乱が見えるかを確認。

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データのplot

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/PSD_wpt_sgi'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='wpt', no_update=True, get_support_data=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='sgi', no_update=True, get_support_data=True)

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

E64_data_wpt_u  = pt.data_quants['erg_pwe_efd_l2_E64Hz_wpt_Eu_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_wpt_v  = pt.data_quants['erg_pwe_efd_l2_E64Hz_wpt_Ev_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = pt.data_quants['erg_pwe_efd_l2_E64Hz_wpt_quality_flag'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

import xarray as xr
import numpy as np

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

# 実際には単純な条件で十分
E64_data_wpt_u_qf = E64_data_wpt_u.where(E64_data_dsi_quality_flag.interp(time=E64_data_wpt_u.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_wpt_v_qf = E64_data_wpt_v.where(E64_data_dsi_quality_flag.interp(time=E64_data_wpt_v.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_wpt = xr.Dataset({
    'E64_wpt_u': E64_data_wpt_u_qf,
    'E64_wpt_v': E64_data_wpt_v_qf
})

ds_E64_wpt  = ds_E64_wpt.dropna(dim='time', how='all')

In [ ]:
B64_data_sgi    = pt.data_quants['erg_mgf_l2_mag_64hz_sgi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_sgi_quality_flag   = pt.data_quants['erg_mgf_l2_quality_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_sgi_quality_flag[:, 3], B64_data_sgi, join='inner')

B64_data_sgi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_sgi  = xr.Dataset({
    'B64_sgi_x':    B64_data_sgi_qf[:, 0],
    'B64_sgi_y':    B64_data_sgi_qf[:, 1],
    'B64_sgi_z':    B64_data_sgi_qf[:, 2]
})

ds_B64_sgi  = ds_B64_sgi.dropna(dim='time', how='all')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_E64_wpt_segs = split_by_gap(ds_E64_wpt, gap_thr=np.timedelta64(8, 's'))
print(len(ds_E64_wpt_segs))

In [ ]:
ds_B64_sgi_segs = split_by_gap(ds_B64_sgi, gap_thr=np.timedelta64(8, 's'))
print(len(ds_B64_sgi_segs))

# Wavelet analysis

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

sys.path.append("..")
import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

vars_E64    = ['E64_wpt_u','E64_wpt_v']
ds_E64_wpt_cwt_seg0 = tw.cwt_from_dataset(ds_E64_wpt_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_wpt_cwt_seg1 = tw.cwt_from_dataset(ds_E64_wpt_segs[1], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_wpt_cwt_seg2 = tw.cwt_from_dataset(ds_E64_wpt_segs[2], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_wpt_cwt_seg3 = tw.cwt_from_dataset(ds_E64_wpt_segs[3], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_wpt_cwt_seg4 = tw.cwt_from_dataset(ds_E64_wpt_segs[4], dt=1/64, s0=2, dj=1/32, variables=vars_E64)

vars_B64    = ['B64_sgi_x','B64_sgi_y', 'B64_sgi_z']
ds_B64_sgi_cwt_seg0 = tw.cwt_from_dataset(ds_B64_sgi_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_B64)

print(ds_E64_wpt_cwt_seg3)
print(ds_B64_sgi_cwt_seg0)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
dsets_E64 = [ds_E64_wpt_cwt_seg0, ds_E64_wpt_cwt_seg1, ds_E64_wpt_cwt_seg2, ds_E64_wpt_cwt_seg3, ds_E64_wpt_cwt_seg4]
targets = [
    ("E64_wpt_u_cwt", r"$E_{u}$ (WPT)", "[(mV/m)$^2$/Hz]"),
    ("E64_wpt_v_cwt", r"$E_{v}$ (WPT)", "[(mV/m)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_E64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 4), sharex=True)
    for ax, (v, ylab, unit) in zip(axes, targets):
        if v not in joined: continue
        da, coi = joined[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'E_fields_wpt_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

In [ ]:
dsets_B64 = [ds_B64_sgi_cwt_seg0]
targets = [
    ("B64_sgi_x_cwt", r"$B_{x}$ (SGI)", "[(nT)$^2$/Hz]"),
    ("B64_sgi_y_cwt", r"$B_{y}$ (SGI)", "[(nT)$^2$/Hz]"),
    ("B64_sgi_z_cwt", r"$B_{z}$ (SGI)", "[(nT)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_B64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 6), sharex=True)
    for ax, (v, ylab, unit) in zip(axes, targets):
        if v not in joined: continue
        da, coi = joined[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'B_fields_sgi_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# 調査時刻におけるPSDのMedianを抽出

In [ ]:
dsets_E64 = [ds_E64_wpt_cwt_seg0, ds_E64_wpt_cwt_seg1, ds_E64_wpt_cwt_seg2, ds_E64_wpt_cwt_seg3, ds_E64_wpt_cwt_seg4]
targets = [
    ("E64_wpt_u_cwt", r"$E_{u}$ (WPT)", "[(mV/m)$^2$/Hz]"),
    ("E64_wpt_v_cwt", r"$E_{v}$ (WPT)", "[(mV/m)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_E64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

da_E64_wpt_u_cwt    = joined['E64_wpt_u_cwt']
da_E64_wpt_v_cwt    = joined['E64_wpt_v_cwt']

In [ ]:
dsets_B64 = [ds_B64_sgi_cwt_seg0]
targets = [
    ("B64_sgi_x_cwt", r"$B_{x}$ (SGI)", "[(nT)$^2$/Hz]"),
    ("B64_sgi_y_cwt", r"$B_{y}$ (SGI)", "[(nT)$^2$/Hz]"),
    ("B64_sgi_z_cwt", r"$B_{z}$ (SGI)", "[(nT)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_B64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

da_B64_sgi_x_cwt    = joined['B64_sgi_x_cwt']
da_B64_sgi_y_cwt    = joined['B64_sgi_y_cwt']
da_B64_sgi_z_cwt    = joined['B64_sgi_z_cwt']

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# ---- 入力 ----
pairs = [
    ('E64_wpt_u_cwt', da_E64_wpt_u_cwt),
    ('E64_wpt_v_cwt', da_E64_wpt_v_cwt),
    ('B64_sgi_x_cwt', da_B64_sgi_x_cwt),
    ('B64_sgi_y_cwt', da_B64_sgi_y_cwt),
    ('B64_sgi_z_cwt', da_B64_sgi_z_cwt),
]
t_all_start = np.datetime64('2022-09-01T21:00:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = path_base_save_plot  # 既存の保存先を使用

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_sorted = []
for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    for name, med in mdict.items():
        prefix = name.split('_')[0]  # 'E64' or 'B64'
        comp   = name.split('_')[2]  # 'x','y','z'
        label  = f"${prefix[0]}_{comp}$"
        ax.loglog(med['freq'], med, label=label)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4, 1E5, 1E6, 1E7])
    ax.legend(ncol=3)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 1E7)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# ---- 5分窓でループ ----
t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
for t0 in t_starts:
    t1 = t0 + step
    mdict = compute_median_dict(pairs_sorted, t0, t1)
    if not mdict:  # その窓でデータ無し
        continue
    plot_median_dict(mdict, t0, t1, outdir=outdir)
